# 02 — Terrain Features (100 m² tiles)

Run DEM-derived zonal statistics on the new tiles.
Output: `../data/terrain_100m2.csv`

In [1]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
import geopandas as gpd
import pandas as pd
import numpy as np

DEM_PATH     = '../data/DEM/RegressionRidge_DEM.tif'
DEM_UTM_PATH = '../data/DEM/RegressionRidge_DEM_utm.tif'
TILES_PKL    = '../data/polygons/tiles_100m2.pkl'
OUT_CSV      = '../data/terrain_100m2.csv'


/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
# Reproject DEM to UTM if needed
with rasterio.open(DEM_PATH) as src:
    if src.crs.to_epsg() != 32610:
        dst_crs = 'EPSG:32610'
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update(crs=dst_crs, transform=transform, width=width, height=height)
        with rasterio.open(DEM_UTM_PATH, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.bilinear
                )
        dem_file = DEM_UTM_PATH
    else:
        dem_file = DEM_PATH

print('DEM file:', dem_file)


DEM file: ../data/DEM/RegressionRidge_DEM_utm.tif


In [3]:
with rasterio.open(dem_file) as src:
    dem       = src.read(1).astype(float)
    transform = src.transform
    nodata    = src.nodata
    xres, yres = transform.a, transform.e

dem = np.where(dem == nodata, np.nan, dem)

dy, dx = np.gradient(dem, abs(yres), abs(xres))
dxx, dxy = np.gradient(dx, abs(yres), abs(xres))
dyx, dyy = np.gradient(dy, abs(yres), abs(xres))

slope_deg   = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
aspect_deg  = (np.degrees(np.arctan2(-dx, dy)) + 360) % 360
curvature   = dxx + dyy
profile_curv = ((dxx*dx**2 + 2*dxy*dx*dy + dyy*dy**2)
                / ((dx**2+dy**2) * (1+dx**2+dy**2)**1.5 + 1e-9))
plan_curv    = ((dxx*dy**2 - 2*dxy*dx*dy + dyy*dx**2)
                / ((dx**2+dy**2) * (1+dx**2+dy**2)**0.5 + 1e-9))
elev_dev     = dem - np.nanmean(dem)

print('DEM derivatives computed')


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:14: RuntimeWarning: invalid value encountered in remainder
  


DEM derivatives computed


In [4]:
tiles = pd.read_pickle(TILES_PKL).to_crs(rasterio.open(dem_file).crs)
print(f'Computing zonal stats for {len(tiles)} tiles…')

def zstats(arr, label):
    return zonal_stats(tiles, arr, affine=transform, nodata=np.nan,
                       stats=['mean', 'min', 'max'], prefix=f'{label}_')

stats = {}
for arr, label in [
    (dem,          'elev'),
    (slope_deg,    'slope'),
    (aspect_deg,   'aspect'),
    (curvature,    'curvature'),
    (profile_curv, 'profile_curv'),
    (plan_curv,    'plan_curv'),
    (elev_dev,     'elev_dev'),
]:
    for i, s in enumerate(zstats(arr, label)):
        if i == 0:
            stats[label] = []
        stats[label].append(s)

print('Zonal stats done')


Computing zonal stats for 32978 tiles…
Zonal stats done


In [5]:
dfs = [tiles[['tile_id']].reset_index(drop=True)]
for label, stat_list in stats.items():
    df_s = pd.DataFrame(stat_list)
    df_s.columns = [c.replace(f'{label}_', f'{label}_') for c in df_s.columns]
    dfs.append(df_s)

terrain = pd.concat(dfs, axis=1)

# Encode aspect as sin/cos vectors
terrain['aspect_cos'] = np.cos(np.radians(terrain['aspect_mean']))
terrain['aspect_sin'] = np.sin(np.radians(terrain['aspect_mean']))
terrain['slope_grad'] = np.tan(np.radians(terrain['slope_mean']))
terrain['slope_x']    = terrain['slope_grad'] * terrain['aspect_cos']
terrain['slope_y']    = terrain['slope_grad'] * terrain['aspect_sin']
terrain = terrain.drop(columns=[c for c in terrain.columns
                                if 'aspect_min' in c or 'aspect_max' in c
                                or 'slope_min' in c or 'slope_max' in c])

terrain.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}  shape: {terrain.shape}')
terrain.head()


Saved → ../data/terrain_100m2.csv  shape: (32978, 23)


,tile_id,elev_min,elev_max,elev_mean,slope_mean,aspect_mean,curvature_min,curvature_max,curvature_mean,profile_curv_min,...,plan_curv_max,plan_curv_mean,elev_dev_min,elev_dev_max,elev_dev_mean,aspect_cos,aspect_sin,slope_grad,slope_x,slope_y
0,0,207.326294,207.821213,207.583952,5.428053,179.226246,-0.007596,0.023885,0.003611,-0.004058,...,0.011806,0.001648,22.362454,22.857373,22.620112,-0.999909,0.013504,0.095022,-0.095013,0.001283
1,1,207.337891,207.834198,207.613095,5.296551,180.673006,-0.012949,0.016167,0.001205,-0.006429,...,0.009277,0.000690,22.374050,22.870358,22.649255,-0.999931,-0.011746,0.092706,-0.092700,-0.001089
2,2,207.426865,207.776810,207.591328,4.993705,172.255903,-0.013323,0.012354,-0.001567,-0.010633,...,0.006050,-0.000508,22.463024,22.812970,22.627488,-0.990880,0.134749,0.087378,-0.086581,0.011774
3,3,207.391617,207.676987,207.529359,5.119117,175.722204,-0.014421,0.024709,0.000654,-0.007211,...,0.012059,0.000576,22.427777,22.713147,22.565519,-0.997214,0.074592,0.089584,-0.089334,0.006682
4,4,208.503342,209.164948,208.922363,6.365033,175.098933,-0.013948,0.015268,0.002957,-0.007070,...,0.007820,0.001702,23.539502,24.201107,23.958523,-0.996344,0.085435,0.111550,-0.111142,0.009530
